In [3]:
# Импорт .env для подключения к БД

from dotenv import load_dotenv
from pathlib import Path

env_path = Path.cwd().parent / '.env'
load_dotenv(env_path)
env_path

PosixPath('/home/ilgiz/ml_lessons/sqlalchemy_lessons/.env')

In [4]:
from models import WorkersOrm, ResumeOrm, WorkLoad
from schemas import ResumeDTO, ResumeRelDTO, WorkersDTO, WorkerRelDTO
from database import sync_session_factory, sync_engine

from sqlalchemy import select, func, and_, or_, Integer
from sqlalchemy.orm import selectinload

## Без relationship

In [5]:
with sync_session_factory() as session:
    q = select(WorkersOrm).limit(2)
    print(q)
    
    res_orm = session.execute(q).scalars().all()
    print(f"{res_orm=}")
    
    res_dto = [WorkersDTO.model_validate(row, from_attributes=True) for row in res_orm]
    print(res_dto)

SELECT workers.id, workers.username 
FROM workers
 LIMIT :param_1
2026-03-09 14:50:09,790 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-09 14:50:09,790 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-09 14:50:09,793 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-09 14:50:09,794 INFO sqlalchemy.engine.Engine [raw sql] {}


2026-03-09 14:50:09,795 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-09 14:50:09,796 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-09 14:50:09,800 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-09 14:50:09,803 INFO sqlalchemy.engine.Engine SELECT workers.id, workers.username 
FROM workers 
 LIMIT %(param_1)s::INTEGER
2026-03-09 14:50:09,804 INFO sqlalchemy.engine.Engine [generated in 0.00108s] {'param_1': 2}
2026-03-09 14:50:09,811 INFO sqlalchemy.engine.Engine SELECT workers_1.id AS workers_1_id, resumes.id AS resumes_id, resumes.title AS resumes_title, resumes.compensation AS resumes_compensation, resumes.workload AS resumes_workload, resumes.worker_id AS resumes_worker_id, resumes.created_at AS resumes_created_at, resumes.updated_at AS resumes_updated_at 
FROM workers AS workers_1 JOIN resumes ON workers_1.id = resumes.worker_id AND resumes.workload = %(workload_1)s 
WHERE workers_1.id IN (%(primary_keys_1)s::INTEGER, %(primary_keys_2)s::INTEGE

/tmp/ipykernel_381570/3169974240.py:3: SAWarning: relationship 'WorkersOrm.resumes_parttime' will copy column workers.id to column resumes.worker_id, which conflicts with relationship(s): 'WorkersOrm.resumes' (copies workers.id to resumes.worker_id). If this is not the intention, consider if these relationships should be linked with back_populates, or if viewonly=True should be applied to one or more if they are read-only. For the less common case that foreign key constraints are partially overlapping, the orm.foreign() annotation can be used to isolate the columns that should be written towards.   To silence this warning, add the parameter 'overlaps="resumes"' to the 'WorkersOrm.resumes_parttime' relationship. (Background on this warning at: https://sqlalche.me/e/20/qzyx) (This warning originated from the `configure_mappers()` process, which was invoked automatically in response to a user-initiated operation.)
  print(q)


## С repaltionship

In [6]:
with sync_session_factory() as session:
    q = select(WorkersOrm).options(selectinload(WorkersOrm.resumes)).limit(2)
    print(q)
    
    res_orm = session.execute(q).scalars().all()
    print(f"{res_orm=}")
    
    res_dto = [WorkerRelDTO.model_validate(row, from_attributes=True) for row in res_orm]
    print(f"{res_dto=}")

SELECT workers.id, workers.username 
FROM workers
 LIMIT :param_1
2026-03-09 14:50:11,687 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-09 14:50:11,689 INFO sqlalchemy.engine.Engine SELECT workers.id, workers.username 
FROM workers 
 LIMIT %(param_1)s::INTEGER
2026-03-09 14:50:11,690 INFO sqlalchemy.engine.Engine [generated in 0.00127s] {'param_1': 2}
2026-03-09 14:50:11,694 INFO sqlalchemy.engine.Engine SELECT resumes.worker_id AS resumes_worker_id, resumes.id AS resumes_id, resumes.title AS resumes_title, resumes.compensation AS resumes_compensation, resumes.workload AS resumes_workload, resumes.created_at AS resumes_created_at, resumes.updated_at AS resumes_updated_at 
FROM resumes 
WHERE resumes.worker_id IN (%(primary_keys_1)s::INTEGER, %(primary_keys_2)s::INTEGER)
2026-03-09 14:50:11,694 INFO sqlalchemy.engine.Engine [generated in 0.00094s] {'primary_keys_1': 1, 'primary_keys_2': 2}
2026-03-09 14:50:11,698 INFO sqlalchemy.engine.Engine SELECT workers_1.id AS workers_1_id

## JOIN

In [7]:
from sqlalchemy import cast

In [13]:
from pydantic import field_validator

In [17]:
from pydantic import BaseModel

class WorkLoadAvgCompensation(BaseModel):
    workload: WorkLoad
    avg_comp: int
    
    @classmethod
    @field_validator('avg_compensation', mode='before')
    def convert_avg_compensation_to_str(cls, v):
        if v is None: 
            return v
        else:
            return str(v)

In [19]:
with sync_session_factory() as session:
            q = (
                select(
                    ResumeOrm.workload,
                    cast(func.avg(ResumeOrm.compensation), Integer).label("avg_comp"),
                )
                .select_from(ResumeOrm)
                # .where() same .filter(), .filter_by(id=1, workload='fulltime')
                .filter(
                    and_(
                        ResumeOrm.title.contains("Developer"),
                        ResumeOrm.compensation > 40000,
                    )
                )
                .group_by(ResumeOrm.workload)
                .having(cast(func.avg(ResumeOrm.compensation), Integer) > 70000)
            )

            # print(q.compile(compile_kwargs={"literal_binds": True}))

            res_orm = session.execute(q).all()
            print(f"{res_orm=}")
            
            res_dto = [WorkLoadAvgCompensation.model_validate(row, from_attributes=True) for row in res_orm]
            print(f"{res_dto=}")

2026-03-09 15:08:41,274 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-09 15:08:41,276 INFO sqlalchemy.engine.Engine SELECT resumes.workload, CAST(avg(resumes.compensation) AS INTEGER) AS avg_comp 
FROM resumes 
WHERE (resumes.title LIKE '%%' || %(title_1)s::VARCHAR || '%%') AND resumes.compensation > %(compensation_1)s::INTEGER GROUP BY resumes.workload 
HAVING CAST(avg(resumes.compensation) AS INTEGER) > %(param_1)s::INTEGER
2026-03-09 15:08:41,277 INFO sqlalchemy.engine.Engine [cached since 882.1s ago] {'title_1': 'Developer', 'compensation_1': 40000, 'param_1': 70000}
res_orm=[(<WorkLoad.parttime: 'parttime'>, 210000), (<WorkLoad.fulltime: 'fulltime'>, 220000)]
res_dto=[WorkLoadAvgCompensation(workload=<WorkLoad.parttime: 'parttime'>, avg_comp=210000), WorkLoadAvgCompensation(workload=<WorkLoad.fulltime: 'fulltime'>, avg_comp=220000)]
2026-03-09 15:08:41,280 INFO sqlalchemy.engine.Engine ROLLBACK


## Конвертация `pydantic` -> `SQLAlchemy`

In [22]:
res_dto

[WorkLoadAvgCompensation(workload=<WorkLoad.parttime: 'parttime'>, avg_comp=210000),
 WorkLoadAvgCompensation(workload=<WorkLoad.fulltime: 'fulltime'>, avg_comp=220000)]

In [25]:
type(res_dto[0]), res_dto[0]

(__main__.WorkLoadAvgCompensation,
 WorkLoadAvgCompensation(workload=<WorkLoad.parttime: 'parttime'>, avg_comp=210000))

In [28]:
res_dto_dict = [dto_obj.model_dump(exclude={'id'}) for dto_obj in res_dto]
res_dto_dict

[{'workload': <WorkLoad.parttime: 'parttime'>, 'avg_comp': 210000},
 {'workload': <WorkLoad.fulltime: 'fulltime'>, 'avg_comp': 220000}]

In [ ]:
# Далее нужно просто распокавать, указав нужную структуру
# new_resume = ResumeOrm(**res_dto_dict[0])